In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Forecasting libraries
from prophet import Prophet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy import stats

import logging

logging.basicConfig(level=logging.INFO, 
                   format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


class SalesDataProcessor:
    """Process and clean sales data"""
    
    def __init__(self):
        self.scaler = StandardScaler()
        logger.info("SalesDataProcessor initialized")
    
    def generate_sample_data(self, n_records: int = 10000) -> pd.DataFrame:
        """Generate sample sales data for demonstration"""
        logger.info(f"Generating {n_records} sample sales records")
        
        np.random.seed(42)
        
        # Date range
        dates = pd.date_range(start='2022-01-01', end='2023-12-31', freq='D')
        
        # Products
        products = ['Coca-Cola', 'Sprite', 'Fanta', 'Dasani Water', 'Minute Maid']
        regions = ['Nairobi', 'Mombasa', 'Kisumu', 'Nakuru', 'Eldoret']
        
        data = []
        for date in dates:
            for product in products:
                for region in regions:
                    # Base sales with seasonality
                    base_sales = np.random.randint(500, 2000)
                    
                    # Add seasonality (higher in hot months)
                    month = date.month
                    seasonal_factor = 1.0 + (0.3 if month in [1, 2, 12] else 0)
                    
                    # Add day of week effect
                    dow_factor = 1.2 if date.dayofweek in [5, 6] else 1.0
                    
                    # Calculate final quantity
                    quantity = int(base_sales * seasonal_factor * dow_factor)
                    
                    # Calculate revenue
                    unit_price = np.random.uniform(30, 80)
                    revenue = quantity * unit_price
                    
                    data.append({
                        'date': date,
                        'product': product,
                        'region': region,
                        'quantity_sold': quantity,
                        'unit_price': unit_price,
                        'revenue': revenue,
                        'distribution_point_id': np.random.randint(1, 201)
                    })
        
        df = pd.DataFrame(data)
        logger.info(f"Generated {len(df)} sales records")
        return df
    
    def clean_data(self, df: pd.DataFrame) -> pd.DataFrame:
        """Clean and prepare sales data"""
        logger.info("Cleaning sales data")
        
        # Remove duplicates
        initial_count = len(df)
        df = df.drop_duplicates()
        logger.info(f"Removed {initial_count - len(df)} duplicates")
        
        # Handle missing values
        df['quantity_sold'].fillna(df['quantity_sold'].median(), inplace=True)
        df['revenue'].fillna(df['revenue'].median(), inplace=True)
        
        # Remove outliers (using IQR method)
        for col in ['quantity_sold', 'revenue']:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            df = df[(df[col] >= Q1 - 1.5*IQR) & (df[col] <= Q3 + 1.5*IQR)]
        
        # Feature engineering
        df['year'] = df['date'].dt.year
        df['month'] = df['date'].dt.month
        df['quarter'] = df['date'].dt.quarter
        df['day_of_week'] = df['date'].dt.dayofweek
        df['week_of_year'] = df['date'].dt.isocalendar().week
        df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
        
        logger.info(f"Data cleaned: {len(df)} records remaining")
        return df
    
    def aggregate_daily_sales(self, df: pd.DataFrame) -> pd.DataFrame:
        """Aggregate sales by day"""
        daily_sales = df.groupby('date').agg({
            'quantity_sold': 'sum',
            'revenue': 'sum'
        }).reset_index()
        
        daily_sales.columns = ['ds', 'quantity', 'y']
        return daily_sales


class ProphetForecaster:
    """Time series forecasting using Facebook Prophet"""
    
    def __init__(self):
        self.model = None
        self.forecast = None
        logger.info("ProphetForecaster initialized")
    
    def prepare_data(self, df: pd.DataFrame, 
                     date_col: str = 'date', 
                     target_col: str = 'revenue') -> pd.DataFrame:
        """Prepare data in Prophet format"""
        prophet_df = df[[date_col, target_col]].copy()
        prophet_df.columns = ['ds', 'y']
        prophet_df['ds'] = pd.to_datetime(prophet_df['ds'])
        return prophet_df
    
    def train(self, df: pd.DataFrame):
        """Train Prophet model"""
        logger.info("Training Prophet model")
        
        self.model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
            seasonality_mode='multiplicative',
            changepoint_prior_scale=0.05
        )
        
        # Add custom seasonalities
        self.model.add_seasonality(
            name='monthly',
            period=30.5,
            fourier_order=5
        )
        
        self.model.fit(df)
        logger.info("Prophet model trained successfully")
    
    def predict(self, periods: int = 30) -> pd.DataFrame:
        """Generate forecast"""
        logger.info(f"Generating {periods}-day forecast")
        
        future = self.model.make_future_dataframe(periods=periods)
        self.forecast = self.model.predict(future)
        
        return self.forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
    
    def evaluate(self, test_df: pd.DataFrame) -> dict:
        """Evaluate model performance"""
        logger.info("Evaluating Prophet model")
        
        # Predict on test set
        forecast = self.model.predict(test_df[['ds']])
        
        # Calculate metrics
        mae = mean_absolute_error(test_df['y'], forecast['yhat'])
        rmse = np.sqrt(mean_squared_error(test_df['y'], forecast['yhat']))
        mape = mean_absolute_percentage_error(test_df['y'], forecast['yhat'])
        
        accuracy = (1 - mape) * 100
        
        metrics = {
            'mae': round(mae, 2),
            'rmse': round(rmse, 2),
            'mape': round(mape * 100, 2),
            'accuracy': round(accuracy, 2)
        }
        
        logger.info(f"Model Accuracy: {metrics['accuracy']}%")
        return metrics
    
    def plot_forecast(self, save_path: str = None):
        """Plot forecast results"""
        if self.forecast is None:
            logger.error("No forecast available. Run predict() first.")
            return
        
        fig = self.model.plot(self.forecast, figsize=(12, 6))
        plt.title('Sales Forecast - Prophet Model', fontsize=14, fontweight='bold')
        plt.xlabel('Date')
        plt.ylabel('Revenue')
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            logger.info(f"Forecast plot saved to {save_path}")
        
        plt.tight_layout()
        plt.show()


class MLForecaster:
    """Machine Learning based forecasting"""
    
    def __init__(self, model_type='random_forest'):
        self.model_type = model_type
        self.model = None
        self.scaler = StandardScaler()
        logger.info(f"MLForecaster initialized with {model_type}")
    
    def create_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Create features for ML model"""
        df = df.copy()
        
        # Time-based features
        df['year'] = df['date'].dt.year
        df['month'] = df['date'].dt.month
        df['day'] = df['date'].dt.day
        df['dayofweek'] = df['date'].dt.dayofweek
        df['quarter'] = df['date'].dt.quarter
        df['dayofyear'] = df['date'].dt.dayofyear
        df['weekofyear'] = df['date'].dt.isocalendar().week
        
        # Lag features
        for lag in [1, 7, 14, 30]:
            df[f'lag_{lag}'] = df['revenue'].shift(lag)
        
        # Rolling statistics
        for window in [7, 14, 30]:
            df[f'rolling_mean_{window}'] = df['revenue'].rolling(window=window).mean()
            df[f'rolling_std_{window}'] = df['revenue'].rolling(window=window).std()
        
        # Drop NaN values created by lag and rolling
        df = df.dropna()
        
        return df
    
    def train(self, X_train, y_train):
        """Train ML model"""
        logger.info(f"Training {self.model_type} model")
        
        # Scale features
        X_train_scaled = self.scaler.fit_transform(X_train)
        
        # Initialize model
        if self.model_type == 'random_forest':
            self.model = RandomForestRegressor(
                n_estimators=100,
                max_depth=10,
                random_state=42,
                n_jobs=-1
            )
        elif self.model_type == 'gradient_boosting':
            self.model = GradientBoostingRegressor(
                n_estimators=100,
                max_depth=5,
                learning_rate=0.1,
                random_state=42
            )
        
        # Train model
        self.model.fit(X_train_scaled, y_train)
        
        # Cross-validation
        cv_scores = cross_val_score(
            self.model, X_train_scaled, y_train, 
            cv=5, scoring='neg_mean_absolute_error'
        )
        
        logger.info(f"Cross-validation MAE: {-cv_scores.mean():.2f} (+/- {cv_scores.std():.2f})")
    
    def predict(self, X_test):
        """Make predictions"""
        X_test_scaled = self.scaler.transform(X_test)
        predictions = self.model.predict(X_test_scaled)
        return predictions
    
    def evaluate(self, X_test, y_test) -> dict:
        """Evaluate model"""
        logger.info("Evaluating ML model")
        
        predictions = self.predict(X_test)
        
        mae = mean_absolute_error(y_test, predictions)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        mape = mean_absolute_percentage_error(y_test, predictions)
        
        accuracy = (1 - mape) * 100
        
        metrics = {
            'mae': round(mae, 2),
            'rmse': round(rmse, 2),
            'mape': round(mape * 100, 2),
            'accuracy': round(accuracy, 2)
        }
        
        logger.info(f"Model Accuracy: {metrics['accuracy']}%")
        return metrics
    
    def feature_importance(self, feature_names) -> pd.DataFrame:
        """Get feature importance"""
        if hasattr(self.model, 'feature_importances_'):
            importance_df = pd.DataFrame({
                'feature': feature_names,
                'importance': self.model.feature_importances_
            }).sort_values('importance', ascending=False)
            
            return importance_df
        return None


class RevenueOptimizer:
    """Identify revenue opportunities"""
    
    def __init__(self):
        logger.info("RevenueOptimizer initialized")
    
    def identify_opportunities(self, df: pd.DataFrame) -> pd.DataFrame:
        """Identify revenue optimization opportunities"""
        logger.info("Identifying revenue opportunities")
        
        # Calculate regional benchmarks
        regional_benchmark = df.groupby('region')['revenue'].agg([
            ('avg_revenue', 'mean'),
            ('median_revenue', 'median'),
            ('total_revenue', 'sum')
        ]).reset_index()
        
        # Calculate product benchmarks
        product_benchmark = df.groupby('product')['revenue'].agg([
            ('avg_revenue', 'mean'),
            ('total_revenue', 'sum')
        ]).reset_index()
        
        # Identify underperforming regions
        opportunities = df.groupby(['region', 'product'])['revenue'].sum().reset_index()
        opportunities = opportunities.merge(
            product_benchmark[['product', 'avg_revenue']], 
            on='product', 
            suffixes=('', '_benchmark')
        )
        
        # Calculate gap
        opportunities['revenue_gap'] = opportunities['avg_revenue'] - opportunities['revenue']
        opportunities['opportunity_value'] = opportunities['revenue_gap'].clip(lower=0)
        
        # Calculate total opportunity
        total_opportunity = opportunities['opportunity_value'].sum()
        
        logger.info(f"Total revenue opportunity identified: ${total_opportunity:,.2f}")
        
        return opportunities.sort_values('opportunity_value', ascending=False)
    
    def optimize_inventory(self, forecast_df: pd.DataFrame) -> pd.DataFrame:
        """Optimize inventory based on forecast"""
        logger.info("Optimizing inventory levels")
        
        # Calculate optimal stock levels
        forecast_df['recommended_stock'] = forecast_df['yhat'] * 1.2  # 20% buffer
        forecast_df['safety_stock'] = forecast_df['yhat'] * 0.3  # 30% safety stock
        forecast_df['reorder_point'] = forecast_df['yhat'] * 0.7  # Reorder at 70%
        
        return forecast_df


class ForecastingPipeline:
    """Complete forecasting pipeline"""
    
    def __init__(self):
        self.processor = SalesDataProcessor()
        self.prophet_model = ProphetForecaster()
        self.ml_model = MLForecaster('random_forest')
        self.optimizer = RevenueOptimizer()
        logger.info("ForecastingPipeline initialized")
    
    def run_complete_analysis(self):
        """Run complete forecasting analysis"""
        logger.info("=" * 60)
        logger.info("STARTING SALES FORECASTING PIPELINE")
        logger.info("=" * 60)
        
        # Step 1: Generate/Load Data
        logger.info("\n--- STEP 1: Data Processing ---")
        sales_data = self.processor.generate_sample_data(n_records=50000)
        sales_clean = self.processor.clean_data(sales_data)
        
        # Step 2: Prepare data for Prophet
        logger.info("\n--- STEP 2: Prophet Forecasting ---")
        daily_sales = self.processor.aggregate_daily_sales(sales_clean)
        
        # Split data
        train_size = int(len(daily_sales) * 0.8)
        train_data = daily_sales[:train_size]
        test_data = daily_sales[train_size:]
        
        # Train Prophet
        self.prophet_model.train(train_data)
        prophet_metrics = self.prophet_model.evaluate(test_data)
        
        # Generate forecast
        forecast = self.prophet_model.predict(periods=30)
        
        # Step 3: ML Model
        logger.info("\n--- STEP 3: ML Forecasting ---")
        
        # Prepare features
        ml_data = sales_clean.groupby('date').agg({
            'revenue': 'sum'
        }).reset_index()
        
        ml_features = self.ml_model.create_features(ml_data)
        
        # Split features
        feature_cols = [col for col in ml_features.columns 
                       if col not in ['date', 'revenue']]
        X = ml_features[feature_cols]
        y = ml_features['revenue']
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, shuffle=False
        )
        
        # Train ML model
        self.ml_model.train(X_train, y_train)
        ml_metrics = self.ml_model.evaluate(X_test, y_test)
        
        # Feature importance
        feature_importance = self.ml_model.feature_importance(feature_cols)
        
        # Step 4: Revenue Optimization
        logger.info("\n--- STEP 4: Revenue Optimization ---")
        opportunities = self.optimizer.identify_opportunities(sales_clean)
        optimized_inventory = self.optimizer.optimize_inventory(forecast)
        
        # Step 5: Results Summary
        logger.info("\n" + "=" * 60)
        logger.info("FORECASTING PIPELINE RESULTS")
        logger.info("=" * 60)
        
        print(f"\nProphet Model Performance:")
        print(f"  Accuracy: {prophet_metrics['accuracy']}%")
        print(f"  MAE: ${prophet_metrics['mae']:,.2f}")
        print(f"  RMSE: ${prophet_metrics['rmse']:,.2f}")
        
        print(f"\nML Model Performance:")
        print(f"  Accuracy: {ml_metrics['accuracy']}%")
        print(f"  MAE: ${ml_metrics['mae']:,.2f}")
        print(f"  RMSE: ${ml_metrics['rmse']:,.2f}")
        
        print(f"\nRevenue Opportunities:")
        print(f"  Total Opportunity: ${opportunities['opportunity_value'].sum():,.2f}")
        print(f"  Number of Opportunities: {len(opportunities[opportunities['opportunity_value'] > 0])}")
        
        print("\nTop 5 Revenue Opportunities:")
        print(opportunities[['region', 'product', 'opportunity_value']].head())
        
        print("\nTop 5 Important Features:")
        if feature_importance is not None:
            print(feature_importance.head())
        
        logger.info("\n" + "=" * 60)
        logger.info("PIPELINE COMPLETED SUCCESSFULLY")
        logger.info("=" * 60)
        
        return {
            'prophet_metrics': prophet_metrics,
            'ml_metrics': ml_metrics,
            'forecast': forecast,
            'opportunities': opportunities,
            'feature_importance': feature_importance
        }


# Main execution
if __name__ == "__main__":
    # Initialize and run pipeline
    pipeline = ForecastingPipeline()
    results = pipeline.run_complete_analysis()
    
    # Additional visualization
    print("\nGenerating forecast visualization...")
    pipeline.prophet_model.plot_forecast('sales_forecast.png')

ModuleNotFoundError: No module named 'pandas'